In [1]:
import polars as pl
import altair as alt
import functions

In [2]:
# Generate dataset using the function with treatment impact
df = functions.generate_dataset(num_units=60, num_obs=1000, impact=0.0, distr="geom")

In [3]:
unit_summary = df.group_by("unit_id").agg(
    [
        pl.len().alias("num_observations"),
        pl.first("base_success_rate").alias("base_success_rate"),
        pl.mean("outcome").alias("observed_success_rate"),
    ]
)

# Sort unit_summary by num_observations descending
unit_summary_sorted = unit_summary.sort("num_observations", descending=True)

# Create bar chart
# Calculate percent of total observations for each unit
total_obs = unit_summary_sorted["num_observations"].sum()
unit_summary_sorted = unit_summary_sorted.with_columns(
    (pl.col("num_observations") / total_obs * 100).alias("percent_of_total")
)

chart = (
    alt.Chart(unit_summary_sorted)
    .mark_bar()
    .encode(
        x=alt.X("unit_id:O", sort=None, title="Unit ID"),
        y=alt.Y("percent_of_total:Q", title="% of Total Observations"),
    )
    .properties(title="Percent of Total Observations per Unit (Sorted)")
)

chart

alt.Chart(...)

In [4]:
from tqdm.notebook import tqdm

estimates = []
for i in tqdm(range(500)):
    df = functions.generate_dataset(num_units=60, num_obs=3000, impact=0.0, distr="geom")
    r = functions.analyze_treatment_effect(df)
    [d.update({"i": i}) for d in r]
    estimates.extend(r)

  0%|          | 0/500 [00:00<?, ?it/s]

In [5]:
pl.DataFrame(estimates).group_by("model").agg(
    [
        pl.mean("estimate").alias("mean_estimate"),
        pl.mean("significant").alias("power"),
        # confidence interval width
        (pl.col("ci_upper") - pl.col("ci_lower")).mean().alias("mean_ci_width"),
        (pl.col("ci_upper") - pl.col("ci_lower")).median().alias("median_ci_width"),
    ]
)

model,mean_estimate,power,mean_ci_width,median_ci_width
str,f64,f64,f64,f64
"""bootstrap_optimized""",-0.001396,0.058,0.096326,0.095894
"""unit_level_ols_unweighted""",0.001804,0.032,0.154335,0.154816
"""obs_level_randomization""",-0.000489,0.044,0.07224,0.072133
"""clustered_ols""",-0.001492,0.054,0.096725,0.095646
"""delta_method""",-0.001492,0.054,0.097591,0.096537
